# Module 5 — affective specificity, same-layer mediation, and latent localization

Closes the three gaps both COLM reviewers named, and adds the positive-mechanism result they said the paper needs.

| Part | Question | Reviewer objection it closes |
|---|---|---|
| **M5.A** | Do sad/angry produce the same refusal→fabrication shift? | *"desperation may just be negative affect"* |
| **M5.B** | Does mediation replicate at the extraction layer? | *"L28 vs L21; 15.2≈15.3 could be coincidence"* |
| **M5.C** | Which latent actually carries the +15.4 direct effect? | *"identifies what doesn't cause fabrication, not what does"* |

**Run order matters.** M5.A first — it is cheapest, needs no SAE, and if sad/angry match desperation the paper's framing changes before you build on it.

**Why not reuse `m4_mediation.ipynb`:** that notebook captures the mediator at the *last prompt token* at α=0.3. The unknown-entity latent fires on the entity mention, so last-token capture reads ≈0 and the decomposition collapses. M5 uses `_entity_position` at α=0.5 — the configuration that produced the paper's numbers.

All artifacts sync to the HF dataset repo as they are produced, so a Colab disconnect costs at most one checkpoint interval.

## Cell 1 — environment

In [ ]:
import os
from google.colab import userdata

os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
try:
    os.environ["HF_MODEL_TOKEN"] = userdata.get("HF_MODEL_TOKEN")
except Exception:
    pass  # only needed for the gated Llama weights

!git clone https://github.com/BraydenFeng/Algoverse.git /content/Algoverse 2>/dev/null || (cd /content/Algoverse && git pull)
%cd /content/Algoverse
!pip install -q -r requirements.txt sae_lens

## Cell 2 — load Gemma

M5.A needs only the model. Load the SAE later (Cell 5) so a specificity-only session does not pay for it.

In [ ]:
import sys
sys.path.insert(0, ".")

from src.lib.config import load_config
from src.lib.model_load import load_gemma

cfg = load_config()
EXTRACTION_LAYER = cfg["models"]["primary"]["extraction_layer"]
SAE_LAYER = cfg["sae"]["layer"]
ALPHA = 0.5  # paper headline: capability-preserving peak in Gemma

model, tokenizer = load_gemma(variant="primary")
print(f"extraction layer L{EXTRACTION_LAYER}  |  SAE layer L{SAE_LAYER}  |  alpha={ALPHA}")

## Cell 3 — M5.A smoke test (~5 min)

Five arms × 40 prompts. Confirms steering, classification, and HF sync all work before committing hours. Check that `m5_specificity/L28_a0.5/verdict.txt` appears on the Hub.

In [ ]:
from src.m5_affect_specificity import run_specificity_sweep

smoke = run_specificity_sweep(model, tokenizer, alpha=ALPHA, limit=40, batch_size=8)
smoke

## Cell 4 — M5.A full sweep (~2.5–4 h)

baseline + desperation + sad + angry + calm on the full 2,492-prompt set, one shared `norm_scale`.

**Reading the verdict** (margin = desperation minus the strongest of sad/angry):
- **≥5 pts** → desperation-specific; the affective framing holds.
- **2–5 pts** → partially specific; report every arm.
- **<2 pts** → *not* specific; reframe the title away from "desperation" to negative affect.

The framing call is yours — the module prints numbers and a suggestion, it does not decide.

In [ ]:
specificity = run_specificity_sweep(model, tokenizer, alpha=ALPHA, batch_size=8)
specificity

## Cell 5 — load the SAE (needed from here on)

The PT SAE is where the entity-recognition latents live; it transfers to the IT model (Ferrando §5).

In [ ]:
import json
from pathlib import Path

from src.lib.config import layer_suffix
from src.lib.sae_load import load_sae

sae, sae_cfg, sparsity = load_sae(layer=SAE_LAYER)
sae = sae.to(model.device).eval()

latents_json = Path(cfg["paths"]["outputs_dir"]) / "m4" / layer_suffix(cfg) / "unknown_entity_latents.json"
PUBLISHED_LATENT = json.loads(latents_json.read_text())["top_unknown_entity_latent_idx"]
print(f"SAE d_sae={sae.W_dec.shape[0]}  |  published unknown-entity latent = {PUBLISHED_LATENT}")

## Cell 6 — M5.B: same-layer mediation (~1.5 h)

The published null ran mediation at L21 while behaviour was measured at L28. This reruns all four arms **at the extraction layer** on the same latent, so behaviour and mediation are co-located.

- ACME still ≈0 → the null replicates same-layer and the cross-layer objection is dead.
- ACME materially non-zero → the original null was partly a layer artifact. **That is a finding, not a failure** — report it.

Uses the SAE trained at L21 on residuals from L28. That mismatch is a real caveat: note it, and treat a non-zero result as provisional until checked against an L28-native dictionary.

In [ ]:
from src.m5_mediation import run_mediation

same_layer = run_mediation(
    model, tokenizer, sae,
    latent_idx=PUBLISHED_LATENT,
    layer=EXTRACTION_LAYER,
    alpha=ALPHA,
)
same_layer["rates"]

## Cell 7 — M5.C phase 1: candidate latents (~15 min, forward-only)

Ranks all ~16k latents by how much steering shifts their peak activation. This is *correlational* — it only narrows the search space for the causal screen.

In [ ]:
from src.m5_mediation import rank_candidates

candidates = rank_candidates(
    model, tokenizer, sae,
    layer=SAE_LAYER,
    alpha=ALPHA,
    n_prompts=200,
    top_k=15,
)
candidates

## Cell 8 — M5.C phase 2: THE GO/NO-GO (~2–3 h)

Per candidate, the rescue arm clamps that latent back to its own per-prompt baseline under steering. The drop in fabrication versus the steered arm **is** that latent's ACME. Thresholds are pre-registered — decide before you look:

- **CLEAR** (any latent ≥3 pts) → localizable. Run the full screen; this is the positive-mechanism paper.
- **DISTRIBUTED** (best 1–3 pts) → no single carrier. Pivot to group/subspace mediation — still M5, still this infrastructure.
- **DEAD** (<1 pt) → nothing moves fabrication. Stop; do not spend more on single-latent localization.

A DISTRIBUTED or DEAD result is information, not failure — it says the effect is not carried by any one feature, which is itself publishable with the group-mediation follow-up.

In [ ]:
from src.m5_mediation import screen_latents

screen = screen_latents(
    model, tokenizer, sae,
    candidates=candidates["latent_idx"].tolist(),
    layer=SAE_LAYER,
    alpha=ALPHA,
    n_prompts=150,      # screening subset; confirm the winner on the full set
    threshold_pts=3.0,
)
screen

## Cell 9 — confirm the winner on the full set (only if CLEAR)

Screening ran on 150 prompts. Re-run the top latent on all 2,492 before it goes in a paper.

In [ ]:
winner = int(screen.iloc[0]["latent_idx"])
print(f"confirming latent {winner} on the full set")

confirmed = run_mediation(
    model, tokenizer, sae,
    latent_idx=winner,
    layer=SAE_LAYER,
    alpha=ALPHA,
    tag=f"confirm_L{SAE_LAYER}_a{ALPHA:g}_latent{winner}",
)
confirmed["rates"]

## Artifacts on the Hub

```
m5_specificity/L28_a0.5/{baseline,desperation,sad,angry,calm}.csv
m5_specificity/L28_a0.5/comparison.csv, verdict.txt
m5_mediation/<run_tag>/arm_{A,B,C,D}.csv, decision.txt
m5_screen/L21_a0.5/candidates.csv, screen.csv, verdict.txt
```

**Next, whichever way the screen lands:** a construct-validity pass on any surviving latent — show it fires on epistemic hedges and refusal-to-claim transitions rather than on rare tokens. Both reviewers and the council flagged "the latent is a rare-token detector" as the failure mode that sinks localization claims.